# AE2: manejo de archivos CSV, JSON y TXT

Este notebook **coordina** tres clases, cada una en su propio archivo `.py`:

| Clase | Archivo | Qué hace |
|---|---|---|
| `ManejadorCSV` | `manejador_csv.py` | Importa, modifica y guarda un CSV (`csv.reader` / `csv.writer`) |
| `ManejadorJSON` | `manejador_json.py` | Importa, modifica y guarda un JSON (`json.load` / `json.dump`) |
| `GeneradorResumenTXT` | `generador_resumen_txt.py` | Escribe `resumen.txt` con los resultados reales |

Flujo: descarga manual → importar → modificar → guardar nuevo CSV y JSON → tercera clase → resumen TXT con URL de origen.

## 1. Origen de los datos

Los dos archivos se descargaron **manualmente** de Hugging Face:

- **CSV:** `crispr_delivery.csv`, del dataset `deep-principle/science_biology` (20 registros, 14 columnas). Cada fila describe un caso de edición genética (`USER_MESSAGE`) con puntajes para seis métodos de entrega.
- **JSON:** `questions.json`, del dataset `KeepaceAI/MoveBench` (80 preguntas, en chino, sobre actividad física y salud). Cada registro tiene `problem_id`, un `problem` anidado (`context`, `question`, `choices`), `domain` y `question_format`.

Las URL específicas de origen se guardan en variables para enviarlas después al resumen. Los archivos originales no se modifican: las salidas se guardan con nombres nuevos.

In [1]:
# URL específica de origen de cada archivo (Hugging Face)
URL_CSV = "https://huggingface.co/datasets/deep-principle/science_biology/blob/main/crispr_delivery.csv"
URL_JSON = "https://huggingface.co/datasets/KeepaceAI/MoveBench/blob/main/questions.json"

# Archivos originales descargados (rutas relativas)
RUTA_CSV = "datos/crispr_delivery.csv"
RUTA_JSON = "datos/questions.json"

# Archivos nuevos que se van a generar (los originales se conservan)
SALIDA_CSV = "datos/crispr_delivery_modificado.csv"
SALIDA_JSON = "datos/questions_modificado.json"
SALIDA_TXT = "resumen.txt"

## 2. Importar las clases e inicializar los objetos

In [2]:
from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON
from generador_resumen_txt import GeneradorResumenTXT

manejador_csv = ManejadorCSV(RUTA_CSV, URL_CSV)
manejador_json = ManejadorJSON(RUTA_JSON, URL_JSON)

## 3. CSV: importar, modificar y guardar

In [3]:
manejador_csv.importar()

# La columna "Question" repite un texto muy largo; para mirar los datos mostramos las columnas útiles
COLUMNAS_VISTA = ["USER_MESSAGE", "option_a_score", "option_b_score", "option_c_score",
                  "option_d_score", "option_e_score", "option_f_score"]
print("ANTES de modificar:")
manejador_csv.mostrar(20, columnas=COLUMNAS_VISTA, ancho=50)

ANTES de modificar:
USER_MESSAGE | option_a_score | option_b_score | option_c_score | option_d_score | option_e_score | option_f_score
I hope to edit primary T-cell | 0 | 2 | 2 | 0 | 0 | 1
I hope to edit primar NK cell | 0 | 2 | 2 | 0 | 0 | 1
I hope to edit primary mouse neuron culture | 1 | 2 | 2 | 0 | 0 | 1
I hope to edit microglia cell line | 1 | 2 | 2 | 0 | 0 | 1
I hope to edit primary hematopeotic stem cells | 0 | 2 | 2 | 0 | 0 | 0
I hope to edit primary embyronic stem cells | 0 | 2 | 2 | 0 | 0 | 0
I hope to edit BMDC cells | 0 | 2 | 1 | 0 | 0 | 1
I hope to edit mouse BMDC cells | 0 | 2 | 1 | 0 | 0 | 1
I hope to edit human brain organoid | 0 | 2 | 1 | 0 | 0 | 2
I hope to edit human colon organoid | 0 | 2 | 1 | 0 | 0 | 2
I hope to edit mouse neurons in brain | 0 | 1 | 0 | 0 | 1 | 2
I hope to edit mouse glia cells in brain | 0 | 1 | 0 | 0 | 1 | 2
I hope to edit mouse liver | 1 | 1 | 0 | 0 | 2 | 2
I hope to edit Monkey embryo | 0 | 1 | 2 | 2 | 0 | 0
I hope to edit A549 cell line.  | 

Modificaciones del CSV, todas dentro de métodos de la clase:

1. **Actualizar valores:** corregir tres textos de `USER_MESSAGE` (dos errores de escritura y un espacio sobrante).
2. **Agregar un registro** nuevo, copiando la estructura de la primera fila.
3. **Eliminar un registro** casi duplicado (`BMDC cells` ya está cubierto por `mouse BMDC cells`).

In [4]:
# 1) Actualizar valores: corregir errores de escritura
correcciones = {
    "I hope to edit primar NK cell": "I hope to edit primary NK cell",
    "I hope to edit primary hematopeotic stem cells": "I hope to edit primary hematopoietic stem cells",
    "I hope to edit A549 cell line. ": "I hope to edit A549 cell line",
}
for texto_original, texto_corregido in correcciones.items():
    posicion = manejador_csv.buscar_indice("USER_MESSAGE", texto_original)
    manejador_csv.actualizar_valor(posicion, "USER_MESSAGE", texto_corregido)

# 2) Agregar un registro nuevo (misma estructura que la primera fila)
nuevo = dict(zip(manejador_csv.encabezados, manejador_csv.filas[0]))
nuevo.update({
    "USER_MESSAGE": "I hope to edit zebrafish embryo",
    "option_a_score": "0", "option_b_score": "1", "option_c_score": "1",
    "option_d_score": "2", "option_e_score": "0", "option_f_score": "0",
    "source_ref": "agregado en AE2", "contributor": "estudiante AE2",
})
manejador_csv.agregar_registro([nuevo[columna] for columna in manejador_csv.encabezados])

# 3) Eliminar un registro casi duplicado
posicion = manejador_csv.buscar_indice("USER_MESSAGE", "I hope to edit BMDC cells")
manejador_csv.eliminar_registro(posicion)

print("DESPUÉS de modificar:")
manejador_csv.mostrar(20, columnas=COLUMNAS_VISTA, ancho=50)

DESPUÉS de modificar:
USER_MESSAGE | option_a_score | option_b_score | option_c_score | option_d_score | option_e_score | option_f_score
I hope to edit primary T-cell | 0 | 2 | 2 | 0 | 0 | 1
I hope to edit primary NK cell | 0 | 2 | 2 | 0 | 0 | 1
I hope to edit primary mouse neuron culture | 1 | 2 | 2 | 0 | 0 | 1
I hope to edit microglia cell line | 1 | 2 | 2 | 0 | 0 | 1
I hope to edit primary hematopoietic stem cells | 0 | 2 | 2 | 0 | 0 | 0
I hope to edit primary embyronic stem cells | 0 | 2 | 2 | 0 | 0 | 0
I hope to edit mouse BMDC cells | 0 | 2 | 1 | 0 | 0 | 1
I hope to edit human brain organoid | 0 | 2 | 1 | 0 | 0 | 2
I hope to edit human colon organoid | 0 | 2 | 1 | 0 | 0 | 2
I hope to edit mouse neurons in brain | 0 | 1 | 0 | 0 | 1 | 2
I hope to edit mouse glia cells in brain | 0 | 1 | 0 | 0 | 1 | 2
I hope to edit mouse liver | 1 | 1 | 0 | 0 | 2 | 2
I hope to edit Monkey embryo | 0 | 1 | 2 | 2 | 0 | 0
I hope to edit A549 cell line | 1 | 2 | 1 | 0 | 0 | 0
I hope to edit Hela cell l

In [5]:
manejador_csv.guardar(SALIDA_CSV)
print("CSV nuevo guardado en:", SALIDA_CSV)

# Verificación: se vuelve a abrir el archivo NUEVO y se comprueba que contiene los cambios
verificador_csv = ManejadorCSV(SALIDA_CSV, URL_CSV)
verificador_csv.importar()
col = verificador_csv.encabezados.index("USER_MESSAGE")
mensajes = [fila[col] for fila in verificador_csv.filas]

assert "I hope to edit primary NK cell" in mensajes
assert "I hope to edit primar NK cell" not in mensajes
assert "I hope to edit primary hematopoietic stem cells" in mensajes
assert "I hope to edit A549 cell line" in mensajes
assert "I hope to edit zebrafish embryo" in mensajes
assert "I hope to edit BMDC cells" not in mensajes
assert len(verificador_csv.filas) == len(manejador_csv.filas)
assert verificador_csv.encabezados == manejador_csv.encabezados
print(f"Verificado: el CSV guardado tiene {len(verificador_csv.filas)} registros e incluye todos los cambios.")

# El archivo ORIGINAL debe seguir intacto
original_csv = ManejadorCSV(RUTA_CSV, URL_CSV)
original_csv.importar()
mensajes_originales = [fila[col] for fila in original_csv.filas]
assert len(original_csv.filas) == 20
assert "I hope to edit primar NK cell" in mensajes_originales
print("Verificado: el CSV original sigue intacto (20 registros, con sus errores originales).")

CSV nuevo guardado en: datos/crispr_delivery_modificado.csv
Verificado: el CSV guardado tiene 20 registros e incluye todos los cambios.
Verificado: el CSV original sigue intacto (20 registros, con sus errores originales).


## 4. JSON: importar, modificar y guardar

In [6]:
manejador_json.importar()
print("ANTES de modificar (primeros 2 registros):")
manejador_json.mostrar(2, ancho=60)

ANTES de modificar (primeros 2 registros):
{
  "problem_id": "DA_OQ262",
  "problem": {
    "context": "用户基本信息：男性，29岁，身高175cm，体重88kg（BMI 28.7，属于肥胖），目前暂无明确的长期运动目标...",
    "question": "用户基本信息：男性，29岁，身高175cm，体重88kg（BMI 28.7，属于肥胖），目前暂无明确的长期运动目标...",
    "choices": {}
  },
  "domain": "DA",
  "question_format": "OQ"
}
{
  "problem_id": "DA_OQ501",
  "problem": {
    "context": "用户画像：小明，16岁，高中生，男性。身高体重未记录（BMI未知）。平时没有明确的运动目标，只是偶尔活动一下身体。...",
    "question": "用户画像：小明，16岁，高中生，男性。身高体重未记录（BMI未知）。平时没有明确的运动目标，只是偶尔活动一下身体。...",
    "choices": {}
  },
  "domain": "DA",
  "question_format": "OQ"
}
... total: 80 registros


Modificaciones del JSON, todas dentro de métodos de la clase:

1. **Actualizar un valor anidado** (`problem.context`) del primer registro, marcándolo como revisado.
2. **Agregar un registro** nuevo con la misma estructura anidada.
3. **Eliminar un registro** buscándolo por su `problem_id`.

In [7]:
# 1) Actualizar un valor anidado: marcar el contexto del primer registro como revisado
contexto = manejador_json.registros[0]["problem"]["context"]
manejador_json.actualizar_valor(0, "problem.context", contexto + " [revisado en AE2]")

# 2) Agregar un registro nuevo con la misma estructura que los demás
manejador_json.agregar_registro({
    "problem_id": "AE2_DEMO_001",
    "problem": {
        "context": "Registro agregado como ejemplo en la actividad AE2.",
        "question": "¿Qué ejercicio recomendarías para empezar?",
        "choices": {},
    },
    "domain": "DA",
    "question_format": "OQ",
})

# 3) Eliminar un registro buscándolo por su identificador
posicion = manejador_json.buscar_indice("problem_id", "DA_OQ336")
manejador_json.eliminar_registro(posicion)

print("DESPUÉS de modificar (primeros 2 registros):")
manejador_json.mostrar(2, ancho=60)
print("Último registro (el agregado):")
print(manejador_json.registros[-1])

DESPUÉS de modificar (primeros 2 registros):
{
  "problem_id": "DA_OQ262",
  "problem": {
    "context": "用户基本信息：男性，29岁，身高175cm，体重88kg（BMI 28.7，属于肥胖），目前暂无明确的长期运动目标...",
    "question": "用户基本信息：男性，29岁，身高175cm，体重88kg（BMI 28.7，属于肥胖），目前暂无明确的长期运动目标...",
    "choices": {}
  },
  "domain": "DA",
  "question_format": "OQ"
}
{
  "problem_id": "DA_OQ501",
  "problem": {
    "context": "用户画像：小明，16岁，高中生，男性。身高体重未记录（BMI未知）。平时没有明确的运动目标，只是偶尔活动一下身体。...",
    "question": "用户画像：小明，16岁，高中生，男性。身高体重未记录（BMI未知）。平时没有明确的运动目标，只是偶尔活动一下身体。...",
    "choices": {}
  },
  "domain": "DA",
  "question_format": "OQ"
}
... total: 80 registros
Último registro (el agregado):
{'problem_id': 'AE2_DEMO_001', 'problem': {'context': 'Registro agregado como ejemplo en la actividad AE2.', 'question': '¿Qué ejercicio recomendarías para empezar?', 'choices': {}}, 'domain': 'DA', 'question_format': 'OQ'}


In [8]:
manejador_json.guardar(SALIDA_JSON)
print("JSON nuevo guardado en:", SALIDA_JSON)

# Verificación: se vuelve a abrir el archivo NUEVO y se comprueba que contiene los cambios
verificador_json = ManejadorJSON(SALIDA_JSON, URL_JSON)
verificador_json.importar()
ids = [registro["problem_id"] for registro in verificador_json.registros]

assert verificador_json.registros[0]["problem"]["context"].endswith("[revisado en AE2]")
assert "AE2_DEMO_001" in ids
assert "DA_OQ336" not in ids
assert len(verificador_json.registros) == len(manejador_json.registros)
print(f"Verificado: el JSON guardado tiene {len(ids)} registros e incluye todos los cambios.")

# El archivo ORIGINAL debe seguir intacto
original_json = ManejadorJSON(RUTA_JSON, URL_JSON)
original_json.importar()
ids_originales = [registro["problem_id"] for registro in original_json.registros]
assert len(original_json.registros) == 80
assert "DA_OQ336" in ids_originales
assert not original_json.registros[0]["problem"]["context"].endswith("[revisado en AE2]")
print("Verificado: el JSON original sigue intacto (80 registros, sin cambios).")

JSON nuevo guardado en: datos/questions_modificado.json
Verificado: el JSON guardado tiene 80 registros e incluye todos los cambios.
Verificado: el JSON original sigue intacto (80 registros, sin cambios).


## 5. Resumen TXT con la tercera clase

La tercera clase recibe los resultados de las otras dos (`resumen()`), incluidas las URL de origen, y genera `resumen.txt` mediante código.

In [9]:
generador = GeneradorResumenTXT([manejador_csv.resumen(), manejador_json.resumen()])
generador.guardar(SALIDA_TXT)
print("Resumen guardado en:", SALIDA_TXT)
print()
generador.mostrar()

Resumen guardado en: resumen.txt

RESUMEN DE LA EJECUCIÓN - AE2: Manejo de archivos CSV y JSON
Generado el: 2026-11-07 20:51:11

FORMATO: CSV
------------------------------------------------------------
URL de origen:        https://huggingface.co/datasets/deep-principle/science_biology/blob/main/crispr_delivery.csv
Archivo importado:    datos/crispr_delivery.csv
Archivo guardado:     datos/crispr_delivery_modificado.csv
Clase utilizada:      ManejadorCSV
Métodos utilizados:   importar, mostrar, buscar_indice, actualizar_valor, agregar_registro, eliminar_registro, guardar, resumen
Registros originales: 20
Registros finales:    20
Columnas/claves:      Question, USER_MESSAGE, option_a_score, option_b_score, option_c_score, option_d_score, option_e_score, option_f_score, Domain, Difficulty, Type, Task, source_ref, contributor
Modificaciones realizadas:
  1. Actualizar valor: fila 1, columna 'USER_MESSAGE': 'I hope to edit primar NK cell' -> 'I hope to edit primary NK cell'
  2. Actualiza

## 6. Archivos generados

In [10]:
import os

print("Carpeta datos/:")
for nombre in sorted(os.listdir("datos")):
    print(f"  {nombre}  ({os.path.getsize(os.path.join('datos', nombre)):,} bytes)")
print("Resumen:", SALIDA_TXT, f"({os.path.getsize(SALIDA_TXT):,} bytes)")

Carpeta datos/:
  crispr_delivery.csv  (18,521 bytes)
  crispr_delivery_modificado.csv  (18,529 bytes)
  questions.json  (138,296 bytes)
  questions_modificado.json  (136,797 bytes)
Resumen: resumen.txt (3,133 bytes)
